# Fire-Smoke YOLO11s 训练
数据集 82k 图（Fire/Smoke 两类），来源 ultralytics platform。流程：下载 ndjson → 并行拉图 → 转 YOLO 格式 → 训练。
**运行时：Runtime → Change runtime type → GPU (L4 或 A100)**
然后 Runtime → Run all。

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics
!apt-get install -y -qq aria2 > /dev/null
import ultralytics; print('ultralytics', ultralytics.__version__)

In [ ]:
# 拉取 ndjson（GCS 公开读）
!mkdir -p /content/fire-smoke
!curl -fsSL -o /content/fire-smoke-dataset.ndjson https://storage.googleapis.com/fire-smoke-training-fb1f8e56/fire-smoke-dataset-v2.ndjson && echo "download ok"
!ls -lh /content/fire-smoke-dataset.ndjson

In [ ]:
# 解析 ndjson：写 labels、生成下载清单（剔除 w/h=0 废框）
import json, os

root = '/content/fire-smoke'
for sp in ['train','val','test']:
    os.makedirs(f'{root}/images/{sp}', exist_ok=True)
    os.makedirs(f'{root}/labels/{sp}', exist_ok=True)

manifest = []
stats = {'no_ann':0, 'bad_boxes':0, 'boxes':0, 'labels':0}
with open('/content/fire-smoke-dataset.ndjson') as f:
    header = json.loads(f.readline())
    for line in f:
        o = json.loads(line)
        if o.get('type') != 'image':
            continue
        sp, fname = o['split'], o['file']
        manifest.append((o['url'], f'{sp}/{fname}'))
        boxes = (o.get('annotations') or {}).get('boxes') or []
        lines = []
        for b in boxes:
            cls, x, y, w, h = b
            if w <= 0 or h <= 0:
                stats['bad_boxes'] += 1
                continue
            lines.append(f'{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
            stats['boxes'] += 1
        if not boxes:
            stats['no_ann'] += 1
        stem = os.path.splitext(fname)[0]
        with open(f'{root}/labels/{sp}/{stem}.txt', 'w') as lf:
            lf.write('\n'.join(lines))
        stats['labels'] += 1

with open('/content/aria2.input', 'w') as f:
    for url, rel in manifest:
        sp, fname = rel.split('/')
        f.write(f'{url}\n  out={fname}\n  dir={root}/images/{sp}\n')

print('manifest:', len(manifest), '| stats:', stats)

In [ ]:
# 并行下载 82k 图（约 9.3GB，视带宽 10-25 分钟）
!aria2c -i /content/aria2.input -j 32 -x 4 --max-tries=3 --retry-wait=2 \
  --console-log-level=warn --summary-interval=120 || true
!echo "images downloaded: $(find /content/fire-smoke/images -type f | wc -l)"

In [ ]:
# data.yaml
yaml = """path: /content/fire-smoke
train: images/train
val: images/val
test: images/test
names:
  0: Fire
  1: Smoke
"""
open('/content/fire-smoke/data.yaml','w').write(yaml)
print(yaml)

In [ ]:
# 训练 yolo11s，100 epochs，imgsz 640，自动 batch。L4 约 5-8 小时
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(
    data='/content/fire-smoke/data.yaml',
    epochs=100,
    imgsz=640,
    batch=-1,
    workers=8,
    project='/content/runs',
    name='yolo11s-fire-smoke',
    patience=20,
    plots=True,
)

In [ ]:
# 结果：指标 + 打包权重（左侧文件面板可下载 best.pt）
import glob
from IPython.display import Image, display
r = '/content/runs/yolo11s-fire-smoke'
for p in ['results.png','confusion_matrix_normalized.png','PR_curve.png','val_batch0_pred.jpg']:
    fs = glob.glob(f'{r}/**/{p}', recursive=True)
    if fs: display(Image(fs[0], width=700))
!cd /content/runs && zip -q -r fire-smoke-weights.zip yolo11s-fire-smoke/weights
!ls -lh /content/runs/fire-smoke-weights.zip